# 8.1 Situation calculus and deductive planning: practice notebook

Describe a changing world in logic and plan by proving theorems. Learn situations, fluents and action axioms, see the frame problem and how successor state axioms solve it, and extract a plan from a proof.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Successor state axioms on a plan

In [ ]:
# Successor state axioms for a robot, two rooms A and B, and a bar of gold.
# A state holds the fluents: where the robot is, where the gold is, and whether the robot holds it.
def possible(action, s):
    if action == "Grab":
        return s["gold"] == s["robot"] and not s["holding"]
    if action == "Release":
        return s["holding"]
    return action in ["Go(A)", "Go(B)"]

def result(action, s):
    # At(Robot, x, Result(a, s)) <=> a = Go(x) or (At(Robot, x, s) and a is not a Go action)
    robot = action[3] if action.startswith("Go") else s["robot"]
    # At(Gold, x, Result(a, s)) <=> (Holding(Gold, s) and a = Go(x)) or (At(Gold, x, s) and not (Holding(Gold, s) and a is a Go action))
    gold = robot if s["holding"] and action.startswith("Go") else s["gold"]
    # Holding(Gold, Result(a, s)) <=> a = Grab or (Holding(Gold, s) and a != Release)
    holding = action == "Grab" or (s["holding"] and action != "Release")
    return {"robot": robot, "gold": gold, "holding": holding}

s = {"robot": "A", "gold": "B", "holding": False}
print("S0", s)
plan = ["Go(B)", "Grab", "Go(A)", "Release"]
for i in range(len(plan)):
    if not possible(plan[i], s):
        print("not possible:", plan[i])
        break
    s = result(plan[i], s)
    print("S" + str(i + 1), s, " after", plan[i])

### Finding the plan term by search

In [ ]:
# Planning as finding a situation term: search the terms Result(a, s) breadth first until the goal holds.
def possible(action, s):
    robot, gold, holding = s
    if action == "Grab":
        return gold == robot and not holding
    if action == "Release":
        return holding
    return True

def result(action, s):
    robot, gold, holding = s
    if action.startswith("Go"):
        robot = action[3]
        if holding:
            gold = robot
    if action == "Grab":
        holding = True
    if action == "Release":
        holding = False
    return (robot, gold, holding)

start = ("A", "B", False)            # robot in A, gold in B, not holding
queue = [(start, "S0")]
seen = [start]
while queue:
    s, term = queue.pop(0)
    if s[1] == "A" and not s[2]:     # goal: gold lies in room A
        print("answer: s =", term)
        break
    for a in ["Go(A)", "Go(B)", "Grab", "Release"]:
        if possible(a, s):
            t = result(a, s)
            if t not in seen:
                seen.append(t)
                queue.append((t, "Result(" + a + ", " + term + ")"))

## Your turn

### Task 1: Successor state axioms

Write `result(action, s)` that returns the new state after the action, using the successor state axioms of this chapter. The robot moves with Go(x). If it holds the gold, the gold moves too. Grab makes holding true, Release makes it false. Assume the action is possible.

Hint: Compute each fluent on its own: robot from Go, gold from robot when holding and moving, holding from Grab and Release.

In [ ]:
# A state is a dictionary: "robot" is "A" or "B", "gold" is "A" or "B", "holding" is True or False.
ACTIONS = ["Go(A)", "Go(B)", "Grab", "Release"]

def result(action, s):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("result('Go(B)', {'robot': 'A', 'gold': 'B', 'holding': False})", {'robot': 'B', 'gold': 'B', 'holding': False}), ("result('Grab', {'robot': 'B', 'gold': 'B', 'holding': False})", {'robot': 'B', 'gold': 'B', 'holding': True}), ("result('Go(A)', {'robot': 'B', 'gold': 'B', 'holding': True})", {'robot': 'A', 'gold': 'A', 'holding': True}), ("result('Release', {'robot': 'A', 'gold': 'A', 'holding': True})", {'robot': 'A', 'gold': 'A', 'holding': False})])

### Task 2: Check a plan with possibility axioms

Write `run_plan(s, plan)`. For each action, if it is not possible (Grab needs the robot and the gold in the same room and not holding, Release needs holding, Go is always possible), return None. Otherwise apply it. Return the final state. The starter gives a correct `result`.

Hint: Write possible(action, s) from the possibility axioms, then loop over the plan.

In [ ]:
# A state is a dictionary: "robot" is "A" or "B", "gold" is "A" or "B", "holding" is True or False.
ACTIONS = ["Go(A)", "Go(B)", "Grab", "Release"]

def result(action, s):
    robot = action[3] if action.startswith("Go") else s["robot"]
    gold = robot if s["holding"] and action.startswith("Go") else s["gold"]
    holding = action == "Grab" or (s["holding"] and action != "Release")
    return {"robot": robot, "gold": gold, "holding": holding}

def run_plan(s, plan):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("run_plan({'robot': 'A', 'gold': 'B', 'holding': False}, ['Go(B)', 'Grab', 'Go(A)', 'Release'])", {'robot': 'A', 'gold': 'A', 'holding': False}), ("run_plan({'robot': 'A', 'gold': 'B', 'holding': False}, ['Grab'])", None), ("run_plan({'robot': 'A', 'gold': 'A', 'holding': False}, ['Release'])", None), ("run_plan({'robot': 'A', 'gold': 'B', 'holding': False}, [])", {'robot': 'A', 'gold': 'B', 'holding': False})])

### Task 3: Find the shortest plan

Write `find_plan(s, goal_gold)`: a shortest list of actions after which the gold lies in room goal_gold and is not held. Use breadth first search over states, trying actions in the order of ACTIONS, and skip actions that are not possible. Return [] if the goal already holds.

Hint: Keep a queue of (state, plan) pairs and a list of seen states. Test the goal when you take a state from the queue.

In [ ]:
# A state is a dictionary: "robot" is "A" or "B", "gold" is "A" or "B", "holding" is True or False.
ACTIONS = ["Go(A)", "Go(B)", "Grab", "Release"]

def result(action, s):
    robot = action[3] if action.startswith("Go") else s["robot"]
    gold = robot if s["holding"] and action.startswith("Go") else s["gold"]
    holding = action == "Grab" or (s["holding"] and action != "Release")
    return {"robot": robot, "gold": gold, "holding": holding}

def possible(action, s):
    if action == "Grab":
        return s["gold"] == s["robot"] and not s["holding"]
    if action == "Release":
        return s["holding"]
    return True

def find_plan(s, goal_gold):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("find_plan({'robot': 'A', 'gold': 'B', 'holding': False}, 'A')", ['Go(B)', 'Grab', 'Go(A)', 'Release']), ("find_plan({'robot': 'B', 'gold': 'B', 'holding': False}, 'A')", ['Grab', 'Go(A)', 'Release']), ("find_plan({'robot': 'A', 'gold': 'A', 'holding': False}, 'A')", []), ("find_plan({'robot': 'A', 'gold': 'A', 'holding': True}, 'B')", ['Go(B)', 'Release'])])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Successor state axioms

Test 3 is the ramification: the held gold moves with the robot.

In [ ]:
# A state is a dictionary: "robot" is "A" or "B", "gold" is "A" or "B", "holding" is True or False.
ACTIONS = ["Go(A)", "Go(B)", "Grab", "Release"]

def result(action, s):
    robot = s["robot"]
    if action.startswith("Go"):
        robot = action[3]
    gold = s["gold"]
    if s["holding"] and action.startswith("Go"):
        gold = robot
    holding = action == "Grab" or (s["holding"] and action != "Release")
    return {"robot": robot, "gold": gold, "holding": holding}

In [ ]:
check([("result('Go(B)', {'robot': 'A', 'gold': 'B', 'holding': False})", {'robot': 'B', 'gold': 'B', 'holding': False}), ("result('Grab', {'robot': 'B', 'gold': 'B', 'holding': False})", {'robot': 'B', 'gold': 'B', 'holding': True}), ("result('Go(A)', {'robot': 'B', 'gold': 'B', 'holding': True})", {'robot': 'A', 'gold': 'A', 'holding': True}), ("result('Release', {'robot': 'A', 'gold': 'A', 'holding': True})", {'robot': 'A', 'gold': 'A', 'holding': False})])

### Solution to task 2: Check a plan with possibility axioms

An empty plan returns the start state unchanged.

In [ ]:
# A state is a dictionary: "robot" is "A" or "B", "gold" is "A" or "B", "holding" is True or False.
ACTIONS = ["Go(A)", "Go(B)", "Grab", "Release"]

def result(action, s):
    robot = action[3] if action.startswith("Go") else s["robot"]
    gold = robot if s["holding"] and action.startswith("Go") else s["gold"]
    holding = action == "Grab" or (s["holding"] and action != "Release")
    return {"robot": robot, "gold": gold, "holding": holding}

def possible(action, s):
    if action == "Grab":
        return s["gold"] == s["robot"] and not s["holding"]
    if action == "Release":
        return s["holding"]
    return True

def run_plan(s, plan):
    for action in plan:
        if not possible(action, s):
            return None
        s = result(action, s)
    return s

In [ ]:
check([("run_plan({'robot': 'A', 'gold': 'B', 'holding': False}, ['Go(B)', 'Grab', 'Go(A)', 'Release'])", {'robot': 'A', 'gold': 'A', 'holding': False}), ("run_plan({'robot': 'A', 'gold': 'B', 'holding': False}, ['Grab'])", None), ("run_plan({'robot': 'A', 'gold': 'A', 'holding': False}, ['Release'])", None), ("run_plan({'robot': 'A', 'gold': 'B', 'holding': False}, [])", {'robot': 'A', 'gold': 'B', 'holding': False})])

### Solution to task 3: Find the shortest plan

Breadth first search returns a shortest plan because it looks at all plans of length k before any of length k + 1.

In [ ]:
# A state is a dictionary: "robot" is "A" or "B", "gold" is "A" or "B", "holding" is True or False.
ACTIONS = ["Go(A)", "Go(B)", "Grab", "Release"]

def result(action, s):
    robot = action[3] if action.startswith("Go") else s["robot"]
    gold = robot if s["holding"] and action.startswith("Go") else s["gold"]
    holding = action == "Grab" or (s["holding"] and action != "Release")
    return {"robot": robot, "gold": gold, "holding": holding}

def possible(action, s):
    if action == "Grab":
        return s["gold"] == s["robot"] and not s["holding"]
    if action == "Release":
        return s["holding"]
    return True

def find_plan(s, goal_gold):
    queue = [(s, [])]
    seen = [s]
    while queue:
        state, plan = queue.pop(0)
        if state["gold"] == goal_gold and not state["holding"]:
            return plan
        for a in ACTIONS:
            if possible(a, state):
                t = result(a, state)
                if t not in seen:
                    seen.append(t)
                    queue.append((t, plan + [a]))
    return None

In [ ]:
check([("find_plan({'robot': 'A', 'gold': 'B', 'holding': False}, 'A')", ['Go(B)', 'Grab', 'Go(A)', 'Release']), ("find_plan({'robot': 'B', 'gold': 'B', 'holding': False}, 'A')", ['Grab', 'Go(A)', 'Release']), ("find_plan({'robot': 'A', 'gold': 'A', 'holding': False}, 'A')", []), ("find_plan({'robot': 'A', 'gold': 'A', 'holding': True}, 'B')", ['Go(B)', 'Release'])])